# Fundamentals PCA

Combine income / balance / cashflow → PCA 2D → scatter colored by sector.

In [ ]:
import json as _json
import pathlib
import sys

import duckdb
import numpy as np
import plotly.express as px
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())


def q(sql, params=None):
    with duckdb.connect(DB, read_only=True) as con:
        return con.execute(sql, params or []).df()


def _plotly_template():
    for p in [
        pathlib.Path.home() / '.config/Code/User/settings.json',
        pathlib.Path.home() / 'Library/Application Support/Code/User/settings.json',
        pathlib.Path.home() / 'AppData/Roaming/Code/User/settings.json',
    ]:
        if p.exists():
            try:
                theme = _json.loads(p.read_text()).get('workbench.colorTheme', '')
                return 'plotly_white' if 'light' in theme.lower() else 'plotly_dark'
            except Exception:
                pass
    return 'plotly_dark'


print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


In [2]:
# Latest annual report per ticker from each statement
LATEST = """
    SELECT * FROM {tbl} WHERE variant = 'A'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY "Ticker" ORDER BY "Report Date" DESC) = 1
"""

# Columns that are metadata, not financial metrics
META = {
    'variant',
    'source_id',
    'source',
    'Currency',
    'Fiscal Year',
    'Fiscal Period',
    'Publish Date',
    'Restated Date',
}
# Shares carried from income only to avoid duplicates across statements
SHARES = {'Shares (Basic)', 'Shares (Diluted)'}

inc = q(LATEST.format(tbl='income'))
bal = q(LATEST.format(tbl='balance'))
cf = q(LATEST.format(tbl='cashflow'))


def fin_cols(df, include_shares=False):
    drop = META | {'Ticker', 'Report Date'} | (set() if include_shares else SHARES)
    return [c for c in df.columns if c not in drop]


inc_f = fin_cols(inc, include_shares=True)
bal_f = fin_cols(bal, include_shares=False)
cf_f = fin_cols(cf, include_shares=False)

inc_r = inc[['Ticker'] + inc_f].rename(columns={c: f'inc_{c}' for c in inc_f})
bal_r = bal[['Ticker'] + bal_f].rename(columns={c: f'bal_{c}' for c in bal_f})
cf_r = cf[['Ticker'] + cf_f].rename(columns={c: f'cf_{c}' for c in cf_f})

df = inc_r.merge(bal_r, on='Ticker', how='inner').merge(cf_r, on='Ticker', how='inner')

companies = q('SELECT "Ticker", "Company Name", "IndustryId" FROM companies')
industries = q('SELECT "IndustryId", "Industry", "Sector" FROM industries')
df = df.merge(companies, on='Ticker', how='left').merge(
    industries, on='IndustryId', how='left'
)

df['Sector'] = df['Sector'].fillna('Unknown')
df['Industry'] = df['Industry'].fillna('Unknown')

feature_cols = [c for c in df.columns if c.startswith(('inc_', 'bal_', 'cf_'))]

print(f'{len(df):,} tickers, {len(feature_cols)} features')

4,459 tickers, 58 features


In [3]:
X = df[feature_cols].values.astype(float)

# Sign-preserving log: compresses large value ranges before normalisation
X_log = np.sign(X) * np.log1p(np.abs(X))

# Impute NaN (not reported) with 0
X_imp = SimpleImputer(strategy='constant', fill_value=0).fit_transform(X_log)

# Z-score normalisation per column: x'_ij = (x_ij - μ_j) / σ_j
X_scaled = StandardScaler().fit_transform(X_imp)

pca = PCA(n_components=3, random_state=42)
coords = pca.fit_transform(X_scaled)

df = df.copy()
df['PC1'] = coords[:, 0]
df['PC2'] = coords[:, 1]
df['PC3'] = coords[:, 2]

ev1, ev2, ev3 = pca.explained_variance_ratio_
print(
    f'Variance explained: PC1={ev1:.1%}  PC2={ev2:.1%}  PC3={ev3:.1%}  total={ev1 + ev2 + ev3:.1%}'
)

Variance explained: PC1=30.0%  PC2=9.2%  PC3=4.5%  total=43.7%


In [11]:
tmpl = _plotly_template()

fig2d = px.scatter(
    df,
    x='PC1',
    y='PC2',
    color='Sector',
    hover_data=['Ticker', 'Company Name', 'Industry'],
    title=f'Fundamentals PCA 2D — {len(df):,} tickers — PC1={ev1:.1%}  PC2={ev2:.1%}',
    template=tmpl,
    height=700,
    opacity=0.7,
)
fig2d.update_traces(marker_size=5)
display(fig2d)

--- /home/md/.config/Code/User/settings.json


In [5]:
fig3d = px.scatter_3d(
    df,
    x='PC1',
    y='PC2',
    z='PC3',
    color='Sector',
    hover_data=['Ticker', 'Company Name', 'Industry'],
    title=f'Fundamentals PCA 3D — {len(df):,} tickers — PC1={ev1:.1%}  PC2={ev2:.1%}  PC3={ev3:.1%}',
    template=tmpl,
    height=800,
    opacity=0.7,
)
fig3d.update_traces(marker_size=3)
display(fig3d)

In [6]:
# Market cap = latest close price × diluted shares outstanding
latest_prices = q("""
    SELECT ticker, close
    FROM prices
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY date DESC) = 1
""")

df_cap = df.merge(latest_prices, left_on='Ticker', right_on='ticker', how='left')

# SimFin shares are in actual units; close is in USD → market cap in USD
df_cap['market_cap'] = df_cap['inc_Shares (Diluted)'] * df_cap['close']


def cap_label(mc):
    if mc is None or mc != mc:  # NaN check
        return 'Unknown'
    if mc < 300e6:
        return 'Micro (<$300M)'
    if mc < 2e9:
        return 'Small ($300M–$2B)'
    if mc < 10e9:
        return 'Mid ($2B–$10B)'
    if mc < 200e9:
        return 'Large ($10B–$200B)'
    return 'Mega (>$200B)'


CAP_ORDER = [
    'Micro (<$300M)',
    'Small ($300M–$2B)',
    'Mid ($2B–$10B)',
    'Large ($10B–$200B)',
    'Mega (>$200B)',
    'Unknown',
]
df_cap['Cap'] = df_cap['market_cap'].apply(cap_label)

print(df_cap['Cap'].value_counts().reindex(CAP_ORDER).dropna().to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Cap
Micro (<$300M)         810
Small ($300M–$2B)      761
Mid ($2B–$10B)         683
Large ($10B–$200B)     602
Mega (>$200B)           49
Unknown               1554


In [7]:
CAP_COLORS = {
    'Micro (<$300M)': '#a8dadc',
    'Small ($300M–$2B)': '#457b9d',
    'Mid ($2B–$10B)': '#1d3557',
    'Large ($10B–$200B)': '#e63946',
    'Mega (>$200B)': '#f4a261',
    'Unknown': '#aaaaaa',
}

fig_cap = px.scatter(
    df_cap,
    x='PC1',
    y='PC2',
    color='Cap',
    category_orders={'Cap': CAP_ORDER},
    color_discrete_map=CAP_COLORS,
    hover_data=['Ticker', 'Company Name', 'Sector'],
    title=f'Fundamentals PCA 2D — colored by market cap — {len(df_cap):,} tickers',
    template=tmpl,
    height=700,
    opacity=0.7,
)
fig_cap.update_traces(marker_size=5)
display(fig_cap)